In [ ]:
"""
Full-fledged OCPP 1.6 Charger Simulator

- Sends mandatory BootNotification, Heartbeat, StatusNotification.
- Listens for OCPP 1.6 Central System (<-broker) CALLs and responds appropriately.
- Dummy data for testing/broker emulation.
"""

import asyncio
import websockets
import json
import datetime
import uuid
import random
import nest_asyncio
import sys

# -----------------------------------------
# CONFIG
# -----------------------------------------
uri = "wss://a2c144aa5356.ngrok-free.app/orgA/SLTEST01"
VENDOR = "SimDemo"
MODEL = "SL/01"
CHARGE_POINT_SERIAL = "SLTEST01"
CHARGE_BOX_SERIAL = "DEMOBOX001"

ocpp16_supported_calls = {
    "Reset",
    "RemoteStartTransaction",
    "RemoteStopTransaction",
    "UnlockConnector",
    "ChangeConfiguration",
    "GetConfiguration",
    "DataTransfer",
    "GetDiagnostics",
    "ClearCache",
    "GetLocalListVersion",
    "SendLocalList",
    "StatusNotification",
    "FirmwareStatusNotification",
    "TriggerMessage",
    "ChangeAvailability",
    "ClearChargingProfile",
    "GetCompositeSchedule",
    "SetChargingProfile",
    "GetDiagnostics",
    "DiagnosticsStatusNotification",
    "StartTransaction",
    "StopTransaction",
    "MeterValues",
    "Authorize",
    "BootNotification",
    "Heartbeat",
    "ReserveNow",
    "CancelReservation",
    "UpdateFirmware",
    "SetChargingProfile"
}

# -----------------------------------------
# OCPP Message helpers
# -----------------------------------------

def ocpp_call(action, payload):
    uid = str(uuid.uuid4())
    return json.dumps([2, uid, action, payload]), uid

def ocpp_call_result(uid, payload):
    return json.dumps([3, uid, payload])

def ocpp_call_error(uid, code, description, details=None):
    return json.dumps([4, uid, code, description, details or {}])

def now_iso():
    return datetime.datetime.utcnow().replace(microsecond=0).isoformat() + "Z"

def handle_central_system_call(action, payload, uid):
    """
    Reply with dummy data for each OCPP 1.6 command.
    See: OCPP 1.6 spec
    """
    # Note: In real charger you'd maintain state (transactions, config, etc).
    # Here we just return plausible dummy.
    if action == "Reset":
        return ocpp_call_result(uid, {"status": "Accepted"})
    elif action == "RemoteStartTransaction":
        return ocpp_call_result(uid, {"status": "Accepted"})
    elif action == "RemoteStopTransaction":
        return ocpp_call_result(uid, {"status": "Accepted"})
    elif action == "UnlockConnector":
        return ocpp_call_result(uid, {"status": "Unlocked"})
    elif action == "ChangeConfiguration":
        return ocpp_call_result(uid, {"status": "Accepted"})
    elif action == "GetConfiguration":
        keys = payload.get("key", [])
        # Always return at least one config key
        resp = {
            "configurationKey": [
                {
                    "key": "AllowOfflineTxForUnknownId",
                    "readonly": False,
                    "value": "true"
                }
            ]
        }
        return ocpp_call_result(uid, resp)
    elif action == "DataTransfer":
        return ocpp_call_result(uid, {"status": "Accepted"})  # vendor-specific
    elif action == "ClearCache":
        return ocpp_call_result(uid, {"status": "Accepted"})
    elif action == "GetLocalListVersion":
        return ocpp_call_result(uid, {"listVersion": 1})
    elif action == "SendLocalList":
        return ocpp_call_result(uid, {"status": "Accepted"})
    elif action == "FirmwareStatusNotification":
        return ocpp_call_result(uid, {})  # No response fields
    elif action == "TriggerMessage":
        return ocpp_call_result(uid, {"status": "Accepted"})
    elif action == "ChangeAvailability":
        return ocpp_call_result(uid, {"status": "Accepted"})
    elif action == "ClearChargingProfile":
        return ocpp_call_result(uid, {"status": "Accepted"})
    elif action == "GetCompositeSchedule":
        resp = {
            "status": "Accepted",
            "connectorId": 1,
            "scheduleStart": now_iso(),
            "chargingSchedule": {
                "duration": 86400,
                "chargingRateUnit": "W",
                "chargingSchedulePeriod": [
                    {"startPeriod": 0, "limit": 22000.0},
                ],
                "minChargingRate": 11000.0
            }
        }
        return ocpp_call_result(uid, resp)
    elif action == "SetChargingProfile":
        return ocpp_call_result(uid, {"status": "Accepted"})
    elif action == "GetDiagnostics":
        # In real, you return a URI. Here we fake it.
        return ocpp_call_result(uid, {"fileName": "http://test.fake/diagnosticfile.txt"})
    elif action == "DiagnosticsStatusNotification":
        return ocpp_call_result(uid, {})
    elif action == "Authorize":
        return ocpp_call_result(uid, {
            "idTagInfo": {"status": "Accepted", "expiryDate": now_iso(), "parentIdTag": "parent"}
        })
    elif action == "StartTransaction":
        return ocpp_call_result(uid, {
            "idTagInfo": {"status": "Accepted"},
            "transactionId": random.randint(1000, 9999)
        })
    elif action == "StopTransaction":
        return ocpp_call_result(uid, {
            "idTagInfo": {"status": "Accepted"}
        })
    elif action == "ReserveNow":
        return ocpp_call_result(uid, {"status": "Accepted"})
    elif action == "CancelReservation":
        return ocpp_call_result(uid, {"status": "Accepted"})
    elif action == "UpdateFirmware":
        return ocpp_call_result(uid, {})
    # Handle unknown action
    return ocpp_call_error(uid, "NotSupported", f"Action '{action}' not implemented", {})

def parse_ocpp_message(msg):
    """Returns tuple: (type, uid, action/payload, payload). Fails gracefully."""
    try:
        data = json.loads(msg)
        if not isinstance(data, list):
            return None, None, None, None
        msg_type = data[0]
        if msg_type == 2 and len(data) == 4:
            # CALL: [2, uid, action, payload]
            return 2, data[1], data[2], data[3]
        elif msg_type == 3 and len(data) == 3:
            # CALLRESULT: [3, uid, payload]
            return 3, data[1], None, data[2]
        elif msg_type == 4 and len(data) == 5:
            # CALLERROR: [4, uid, errorCode, errorDescription, errorDetails]
            return 4, data[1], data[2], data[4]
        else:
            return msg_type, None, None, None
    except Exception:
        return None, None, None, None

# -----------------------------------------
# Main event loop for charger simulator
# -----------------------------------------
async def charger_main():
    print(f"Connecting as OCPP 1.6 charger to: {uri}")
    async with websockets.connect(uri, subprotocols=["ocpp1.6"]) as ws:
        print("✅ WebSocket connected.")

        # 1. BootNotification
        boot_payload = {
            "chargePointVendor": VENDOR,
            "chargePointModel": MODEL,
            "chargePointSerialNumber": CHARGE_POINT_SERIAL,
            "chargeBoxSerialNumber": CHARGE_BOX_SERIAL,
            "firmwareVersion": "1.0",
        }
        boot_msg, boot_uid = ocpp_call("BootNotification", boot_payload)
        await ws.send(boot_msg)
        print("📤 BootNotification sent.")
        # Await reply
        while True:
            reply = await ws.recv()
            typ, uid, _, payload = parse_ocpp_message(reply)
            if typ == 3 and uid == boot_uid:
                print(f"📥 BootNotification response: {payload}")
                if payload.get("status") != "Accepted":
                    print("❌ BootNotification not accepted, exiting.")
                    return
                break

        # Schedule periodic OCPP reporting tasks
        asyncio.create_task(send_heartbeat(ws))
        asyncio.create_task(send_status_notifications(ws))
        asyncio.create_task(send_meter_values(ws))

        # Listen/serve loop: respond to incoming OCPP CALLs
        print("Listening for OCPP 1.6 commands from Central System...")
        while True:
            msg = await ws.recv()
            typ, uid, action, payload = parse_ocpp_message(msg)
            if typ == 2 and action:  # CALL from central system
                print(f"⬇️  CALL from central: {action}, payload={payload}")
                response = handle_central_system_call(action, payload, uid)
                await ws.send(response)
                print(f"   ↪️ Responded to {action}")
            else:
                print("⬇️  Non-CALL message or parsing failed.")

# -----------------------------------------
# Periodic reporting tasks
# -----------------------------------------
async def send_heartbeat(ws):
    """Send Heartbeat every 30 seconds."""
    while True:
        try:
            msg, uid = ocpp_call("Heartbeat", {})
            await ws.send(msg)
            print(f"💓 Heartbeat sent ({now_iso()})")
            await asyncio.sleep(30)
        except Exception as e:
            print(f"💓 Heartbeat error: {e}")
            break

async def send_status_notifications(ws):
    """Send StatusNotification every 60 seconds."""
    while True:
        try:
            payload = {
                "connectorId": 1,
                "status": "Available",
                "errorCode": "NoError",
                "timestamp": now_iso()
            }
            msg, uid = ocpp_call("StatusNotification", payload)
            await ws.send(msg)
            print("⚡ StatusNotification sent: Available")
            await asyncio.sleep(60)
        except Exception as e:
            print(f"⚡ StatusNotification error: {e}")
            break

async def send_meter_values(ws):
    """Send dummy MeterValues every 20 seconds."""
    while True:
        try:
            payload = {
                "connectorId": 1,
                "transactionId": 1234,
                "meterValue": [
                    {
                        "timestamp": now_iso(),
                        "sampledValue": [
                            {"value": str(random.randint(10000, 20000)), "measurand": "Energy.Active.Import.Register", "unit": "Wh"}
                        ]
                    }
                ]
            }
            msg, uid = ocpp_call("MeterValues", payload)
            await ws.send(msg)
            print(f"🔢 MeterValues sent.")
            await asyncio.sleep(20)
        except Exception as e:
            print(f"🔢 MeterValues error: {e}")
            break

# -----------------------------------------
# Run the charger simulator
# -----------------------------------------
if __name__ == "__main__":
    # Fix for: "coroutine 'charger_main' was never awaited"
    # If running in an environment (like Jupyter) where an event loop is already running,
    # use nest_asyncio or fallback to asyncio.ensure_future.
    try:
        try:
            nest_asyncio.apply()
            loop = asyncio.get_event_loop()
            if loop.is_running():
                # We're likely in Jupyter/IPython -- schedule task in existing loop
                task = loop.create_task(charger_main())
                # Optionally: wait for task completion if desired (this blocks!)
                # loop.run_until_complete(task)
            else:
                loop.run_until_complete(charger_main())
        except Exception:
            # Fallback: just try regular asyncio.run (for normal python scripts)
            asyncio.run(charger_main())
    except ImportError:
        # nest_asyncio not available; fallback to basic approach
        try:
            asyncio.run(charger_main())
        except RuntimeError as re:
            loop = asyncio.get_event_loop()
            loop.run_until_complete(charger_main())
    except KeyboardInterrupt:
        print("🛑 Charger stopped manually.")
    except Exception as e:
        print(f"❌ Unexpected error: {e}")


Task exception was never retrieved
future: <Task finished name='Task-1' coro=<charger_main() done, defined at C:\Users\amdud\AppData\Local\Temp\ipykernel_10780\3183186826.py:191> exception=ConnectionClosedError(None, None, None)>
Traceback (most recent call last):
  File "C:\Users\amdud\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\websockets\asyncio\connection.py", line 915, in send_context
    await self.drain()
  File "C:\Users\amdud\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\websockets\asyncio\connection.py", line 1068, in drain
    await waiter
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.13_3.13.2288.0_x64__qbz5n2kfra8p0\Lib\asyncio\futures.py", line 286, in __await__
    yield self  # This tells Task to wait for completion.
    ^^^^^^^^^^
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundati

Connecting as OCPP 1.6 charger to: wss://a2c144aa5356.ngrok-free.app/orgA/SLTEST01
✅ WebSocket connected.
📤 BootNotification sent.
📥 BootNotification response: {'currentTime': '2025-10-18T21:10:57.779824+00:00', 'interval': 300, 'status': 'Accepted'}
Listening for OCPP 1.6 commands from Central System...
💓 Heartbeat sent (2025-10-18T21:10:57Z)


C:\Users\amdud\AppData\Local\Temp\ipykernel_10780\2775046062.py:75: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.datetime.utcnow().replace(microsecond=0).isoformat() + "Z"


⚡ StatusNotification sent: Available
🔢 MeterValues sent.
⬇️  Non-CALL message or parsing failed.
⬇️  Non-CALL message or parsing failed.
⬇️  Non-CALL message or parsing failed.
🔢 MeterValues sent.
⬇️  Non-CALL message or parsing failed.
💓 Heartbeat sent (2025-10-18T21:11:27Z)
⬇️  Non-CALL message or parsing failed.
🔢 MeterValues sent.
⬇️  Non-CALL message or parsing failed.
⚡ StatusNotification sent: Available
💓 Heartbeat sent (2025-10-18T21:11:57Z)
🔢 MeterValues sent.
⬇️  Non-CALL message or parsing failed.
⬇️  Non-CALL message or parsing failed.
⬇️  Non-CALL message or parsing failed.
🔢 MeterValues error: received 1012 (service restart); then sent 1012 (service restart)
💓 Heartbeat error: received 1012 (service restart); then sent 1012 (service restart)
⚡ StatusNotification error: received 1012 (service restart); then sent 1012 (service restart)
